# HSPC_MAC · ATLAS Table 3 · T3 Trajectory & T4 Future-fate Proxy

This notebook trains a **new three-modality ATLAS model (Mc/Mu/Ms)**, evaluates **T3 POA/POAUC**, and tests **T4 train-terminal-reference absorption**. It uses the original V4 data filenames but replaces TIDE/CORAL with the current `atlas.model.ATLAS` plus a new independent HSPC trainer.

**Scientific caveat:** T4 target labels are derived from current annotated cell types (`future_map`), not experimental future-cell outcomes. Unlike the old V4 notebook, held-out test cell labels are not used to construct terminal reference populations or tune settings. Reproduced scores are **not directly comparable** to paper Table 3 until the original protocol is verified.

The notebook adds files only under `src/atlas/hspc_table3.py`, `notebooks/table3_hspc/` and `results/table3_hspc/`—it does **not** change your running E1-E4 experiment.

## 1. Environment and training settings

In [1]:
from pathlib import Path
import sys, json
import pandas as pd

PROJECT_ROOT = Path('/data5/zhangye/ATLAS')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from atlas.hspc_table3 import HSPCConfig, DATASET_FILES, load_hspc_data, run_hspc

DATASET_NAME = 'HSPC_MAC'
DATA_ROOT = Path('/data5/zhangye/VelocityBench/data')
OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'table3_hspc'
ANNOTATION_JSON = PROJECT_ROOT / 'configs' / f'{DATASET_NAME}_annotations.json'
QUICK = True     # Smoke test first; change to False for 80/40/60 epoch full training.
DEVICE = 'cuda:0'
SEED = 1234
TOP_N_GENES = 2000

cfg = HSPCConfig(dataset=DATASET_NAME, data_root=str(DATA_ROOT),
                 output_root=str(OUTPUT_ROOT),device=DEVICE,
                 seed=SEED,top_n_genes=TOP_N_GENES,quick=QUICK)
assert ANNOTATION_JSON.exists(), f'Missing {ANNOTATION_JSON}. Extract the package into the ATLAS root first.'
annotations=json.loads(ANNOTATION_JSON.read_text())
print('Data files:', [str(DATA_ROOT/p) for p in DATASET_FILES[DATASET_NAME]])
print('Annotation file:',ANNOTATION_JSON)
print('Epochs:', '2/1/2 (smoke)' if QUICK else '80/40/60 (full)')

Data files: ['/data5/zhangye/VelocityBench/data/8489-MV-1-9060-MV-3_adata_postpro_concat.h5ad', '/data5/zhangye/VelocityBench/data/8489-MV-1-9060-MV-3_adata_atac_postpro_concat.h5ad']
Annotation file: /data5/zhangye/ATLAS/configs/HSPC_MAC_annotations.json
Epochs: 2/1/2 (smoke)


## 2. Inspect data layout and annotation mappings before training

In [2]:
import anndata as ad
for file_name in DATASET_FILES[DATASET_NAME]:
    p = DATA_ROOT / file_name
    if not p.exists():
        raise FileNotFoundError(f'{p} (original V4 HSPC data is not included in the code bundle)')
    raw = ad.read_h5ad(p, backed='r')
    print(p.name, 'cells=',raw.n_obs,'features=',raw.n_vars,
          'layers=',list(raw.layers.keys()),'obs=',list(raw.obs.columns[:12]))
    if cfg.label_key in raw.obs:
        print(raw.obs[cfg.label_key].astype(str).value_counts().head(25))
    if getattr(raw,'file',None) is not None: raw.file.close()
print('Stage map:', annotations['stage_order_map'])
print('Fate map:', annotations['future_map'])
print('Terminals:', annotations['terminal_types'])

8489-MV-1-9060-MV-3_adata_postpro_concat.h5ad cells= 9908 features= 929 layers= ['Ms', 'Mu', 'counts', 'spliced', 'unspliced'] obs= ['total_unspliced', 'total_spliced', 'log1p_total_unspliced', 'log1p_total_spliced', 'fraction_u', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt']
8489-MV-1-9060-MV-3_adata_atac_postpro_concat.h5ad cells= 9908 features= 929 layers= ['Mc'] obs= ['n_counts', 'batch']
Stage map: {'HSC': 0, 'LMPP': 1, 'CMP': 1, 'GMP': 1, 'MEP': 1, 'Prog B': 1, 'Prog DC': 1, 'Prog MK': 1, 'DC': 2, 'Granulocyte': 2, 'Erythrocyte': 2, 'Megakaryocyte': 2, 'Platelet': 2, 'Mast Cells': 2, 'MDP': 1, 'Monocyte': 2, 'M1 Macrophage': 2, 'M2 Macrophage': 2}
Fate map: {'LMPP': 'lymphoid_dc', 'Prog B': 'lymphoid_dc', 'Prog DC': 'lymphoid_dc', 'DC': 'lymphoid_dc', 'CMP': 'myeloid', 'GMP': 'myeloid', 'Granulocyte': 'myeloid', 'Mast Cells': 'myeloid', 'MEP': 'ery_platelet', 'Prog MK

## 3. Train ATLAS (T1/T2 representation → T3 TimeHead + lag → T4 velocity)

The model is trained **without test-stage or test-fate supervision**. Train-stage annotations are used only to decide the otherwise ambiguous time direction during evaluation. Terminal reference labels are taken from training cells only.

This cell performs training and saves separate checkpoints for each phase; it will not overwrite an already completed result directory.

In [ ]:
cfg.label_key = "leiden"

summary, RUN_DIR = run_hspc(cfg, annotations)
print('All outputs:', RUN_DIR)

[HSPC_MAC] cells=9908 genes=929 train=7926 test=1982
Unmapped stage labels: []
Unmapped fate labels: ['HSC']
[HSPC_MAC] phase1 1/2 loss=5.4197
[HSPC_MAC] phase1 2/2 loss=4.2165
[HSPC_MAC] phase2 1/1 loss=5.4949
[teacher] phase3 epoch=1, anchors=4000
[HSPC_MAC] phase3 1/2 loss=4.5597
[HSPC_MAC] phase3 2/2 loss=4.2911


## 4. Read T3 / T4 results and compare with paper reference

The paper values below are reference numbers, **not** automatically reproduced here; label-protocol differences mean direct numeric comparisons are diagnostic only.

In [ ]:
from IPython.display import display, Image
m3=json.loads((RUN_DIR/'T3_metrics.json').read_text())
m4=json.loads((RUN_DIR/'T4_metrics.json').read_text())
paper={'POA':0.61,'POAUC':0.639,'F1':0.769,'AUROC':0.792}
compare=pd.DataFrame({'Paper reference':paper,
                     'This run (heldout/proxy)':{'POA':m3['POA'],'POAUC':m3['POAUC'],
                                                    'F1':m4['F1'],'AUROC':m4['AUROC']}})
display(compare)
print('T3 test n:', m3['n_eval'], 'T4 test n:',m4['n_eval'])
print('T4 time-only baseline: F1 =', m4.get('time_only_F1'), 'AUROC =',m4.get('time_only_AUROC'))
display(Image(filename=str(RUN_DIR/'T3_trajectory.png')))

## 5. Inspect T4 predictions and leakage/coverage diagnostics

In [ ]:
display(pd.read_csv(RUN_DIR / 'T4_heldout_predictions.csv').head(10))
print(json.dumps(json.loads((RUN_DIR / 'T4_transition_diagnostics.json').read_text()),indent=2))
manifest=pd.read_csv(RUN_DIR/'split_manifest.csv')
print('Train:',sum(manifest.partition=='train'),'Test:',sum(manifest.partition=='test'))
assert manifest.cell_id.is_unique
print('T4 reference contains ONLY train terminal cells; test fate labels are scoring-only.')

## 6. Optional: combined table for all HSPC datasets

In [ ]:
# Run this cell after completing all three notebooks.
from subprocess import run
run([sys.executable,str(PROJECT_ROOT/'experiments/summarize_table3_hspc.py'),
     '--root',str(OUTPUT_ROOT)],check=True)